# 运输问题（Transportation）：一个模型一个 Notebook

## 这个模型解决什么问题
有若干产地（供给量固定）和若干销地（需求量固定），从产地 i 运到销地 j 的单位运价是 c_ij，
怎么调运能让**总运费最小**、同时每个产地运空、每个销地满足。

```
min  Σ Σ c_ij · x_ij
s.t. Σ_j x_ij = 供给_i      每个产地运出量 = 产量
     Σ_i x_ij = 需求_j      每个销地收到量 = 需求量
     x_ij >= 0
```

## 供需不平衡怎么办
真实题目经常总供给 ≠ 总需求。本 Notebook 自动补**虚拟产地 / 虚拟销地**（运价 0），
并在诊断里写明补了多少。注意：虚拟节点的运量在论文里要解释成"缺货"或"库存"，
不能当成真实运输量。

## 两种数据输入方式
| 方式 | 数据长什么样 |
| --- | --- |
| 已处理矩阵 | 供给 dict/Series、需求 dict/Series、运价矩阵 DataFrame |
| 原始表（长表） | 三列：产地 / 销地 / 运价，需要先 pivot 成矩阵 |

## 目录
0 环境 / 1 配置 / 2 数据接口 / 3 建模求解 / 4 矩阵示例 / 5 长表示例 / 6 导出

> 依赖 `numpy pandas scipy`（用 `linprog` 解，规模几千个变量也没问题）。


In [1]:
# -*- coding: utf-8 -*-
# =====================================================================
# 0. 环境与路径（每个 Notebook 都带这一段，保证单独打开就能跑）
# =====================================================================
# 依赖：numpy、pandas、scipy（>=1.9，用到 milp）
# 可选：matplotlib（画图）、openpyxl（导出 xlsx）
# 缺哪个就装：pip install numpy pandas scipy matplotlib openpyxl
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize as sp_opt
from scipy.optimize import Bounds, LinearConstraint

try:
    import matplotlib
    import matplotlib.pyplot as plt
    # 不设中文字体，图里的标题和标签会变成方块
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False


def find_root(marker="导入数据"):
    """向上逐级找含 导入数据/ 的目录。

    Notebook 放在 优化类模型/ 下也能找到项目根，换电脑、换目录都不用改死路径。
    """
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).is_dir():
            return p
    return Path.cwd()


ROOT = find_root()
DATA_DIR = ROOT / "导入数据"
OUT_DIR = ROOT / "输出数据"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("项目根目录：", ROOT)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", sp_opt.__name__ and __import__("scipy").__version__)


项目根目录： D:\git项目
numpy 2.4.6 | pandas 3.1.0 | scipy 1.17.1


In [2]:
# =====================================================================
# 1. 配置区
# =====================================================================
CFG = dict(
    tag="运输方案",
    make_plots=True,
    save_excel=True,
)
print("配置：", CFG)


配置： {'tag': '运输方案', 'make_plots': True, 'save_excel': True}


## 2. 数据接口


In [3]:
# =====================================================================
# 2. 数据接口：读文件 → 清洗 → 变成模型能吃的表
# =====================================================================
# 这一格是「原始数据直接放进来」的关键：不用你手动去空格、转数字、去重。
# 如果数据已经处理好，可以跳过清洗，直接用后面的 load_matrix 读矩阵。
#
# ⚠️ 999 是否算缺失取决于题目：若它是真实观测值，把它从 NA_MARKERS 里删掉。
NA_MARKERS = ("", "NA", "N/A", "na", "缺测", "未填", "无", "999", 999, "9999", 9999)


def load_table(data, sheet_name=0, encoding="utf-8-sig"):
    """读 csv / xlsx，也接受现成的 DataFrame。

    - csv 先按 utf-8-sig 读（吃掉 Excel 导出的 BOM），失败再退回 gbk
    - 列名首尾空格自动去掉（" 单耗" → "单耗"）
    """
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        p = Path(data)
        if not p.exists():
            raise FileNotFoundError(f"找不到数据文件：{p}")
        if p.suffix.lower() in (".xlsx", ".xls"):
            df = pd.read_excel(p, sheet_name=sheet_name)
        else:
            try:
                df = pd.read_csv(p, encoding=encoding)
            except UnicodeDecodeError:
                df = pd.read_csv(p, encoding="gbk")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def clean_frame(df, na_markers=NA_MARKERS, drop_missing_rate=50.0,
                fill_num="median", diagnostics=None):
    """把原始附件洗成能建模的表，并把每一步记进 diagnostics。

    顺序有因果，别随意调换：
    ① 哨兵值 → NaN       999 / 缺测 / 未填 统一当缺失
    ② 文本列去空格       否则 "东部" 与 " 东部" 会被当成两个类别
    ③ 文本数字转数值     " 12.5 " 这种带空格的数字
    ④ 删完全重复行
    ⑤ 高缺失列删列       删列不删行，保住样本量
    ⑥ 缺失填补           数值列中位数/均值，文本列众数

    参数
    ----
    diagnostics : 传一个 list 进来，会往里追加每一步的说明，最后可以导出给论文用
    """
    diag = diagnostics if diagnostics is not None else []
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]

    before = int(d.isna().sum().sum())
    d = d.replace(list(na_markers), np.nan)
    diag.append({"步骤": "哨兵值转缺失",
                 "说明": f"新增缺失 {int(d.isna().sum().sum()) - before} 个"})

    obj_cols = [c for c in d.columns
                if pd.api.types.is_string_dtype(d[c]) or pd.api.types.is_object_dtype(d[c])]
    for c in obj_cols:
        d[c] = d[c].astype("string").str.strip()
    if obj_cols:
        diag.append({"步骤": "文本列去空格", "说明": f"处理 {len(obj_cols)} 列"})

    num_cols, cat_cols = [], []
    for c in d.columns:
        s = d[c]
        if pd.api.types.is_numeric_dtype(s):
            num_cols.append(c)
            continue
        conv = pd.to_numeric(s.astype(str).str.strip(), errors="coerce")
        if conv.notna().sum() / max(s.notna().sum(), 1) > 0.9:
            d[c] = conv
            num_cols.append(c)
        else:
            cat_cols.append(c)
    diag.append({"步骤": "列类型识别",
                 "说明": f"数值列 {len(num_cols)} 个、类别列 {len(cat_cols)} 个"})

    n_dup = int(d.duplicated().sum())
    d = d.drop_duplicates(keep="first").reset_index(drop=True)
    diag.append({"步骤": "去完全重复", "说明": f"删除 {n_dup} 行"})

    rate = d.isna().mean() * 100
    drop = list(rate[rate >= drop_missing_rate].index)
    if drop:
        d = d.drop(columns=drop)
        diag.append({"步骤": "删除高缺失列",
                     "说明": f"删除 {drop}（缺失率 ≥ {drop_missing_rate}%），删列不删行"})

    filled = []
    for c in num_cols:
        if c in d.columns and d[c].isna().any():
            fill = {"median": d[c].median(), "mean": d[c].mean(), "zero": 0}[fill_num]
            d[c] = d[c].fillna(fill)
            filled.append(c)
    for c in cat_cols:
        if c in d.columns and d[c].isna().any():
            m = d[c].mode(dropna=True)
            d[c] = d[c].fillna(m.iat[0] if len(m) else "未知")
            filled.append(c)
    if filled:
        diag.append({"步骤": "缺失填补", "说明": f"填补 {filled}，数值方式 {fill_num}"})

    diag.append({"步骤": "清洗完成", "说明": f"{d.shape[0]} 行 × {d.shape[1]} 列"})
    return d


def pivot_long(df, index, columns, values, agg="sum"):
    """长表转宽表：产品-资源-单耗 三列 → 产品 × 资源 矩阵。

    真实附件常见这种长表；同一 (产品, 资源) 出现多次时按 agg 聚合。
    """
    wide = df.pivot_table(index=index, columns=columns, values=values, aggfunc=agg).reset_index()
    wide.columns = [str(c) for c in wide.columns]
    return wide


def load_matrix(data, sheet_name=0):
    """读成本矩阵 / 距离矩阵：第一列当行名，其余列转成数值。"""
    df = load_table(data, sheet_name)
    if df.shape[1] > 1 and not pd.api.types.is_numeric_dtype(df.iloc[:, 0]):
        df = df.set_index(df.columns[0])
    df.index = [str(i) for i in df.index]
    df.columns = [str(c) for c in df.columns]
    return df.astype(float)


## 3. 建模与求解

供需不平衡会自动补虚拟节点，这一步必须写进论文，否则读者会奇怪为什么多了一行/一列。


In [4]:
# =====================================================================
# 3. 建模与求解
# =====================================================================
def to_vector(x, prefix):
    """把 dict / Series / list 统一成 (名称列表, 数值数组)。"""
    if isinstance(x, dict):
        return list(x.keys()), np.array(list(x.values()), dtype=float)
    if isinstance(x, pd.Series):
        return list(x.index), x.to_numpy(dtype=float)
    v = np.asarray(x, dtype=float).ravel()
    return [f"{prefix}{i+1}" for i in range(len(v))], v


def build_transport(supply, demand, cost):
    """运输问题 → 线性规划系数。

    返回 dict（c / A_eq / b_eq / bounds / 名称 / 诊断）。
    供需不平衡时自动补虚拟节点，保证有可行解。
    """
    s_names, s_val = to_vector(supply, "产地")
    d_names, d_val = to_vector(demand, "销地")
    C = cost.to_numpy(dtype=float) if isinstance(cost, pd.DataFrame) else np.asarray(cost, dtype=float)
    if C.shape != (len(s_val), len(d_val)):
        raise ValueError(f"运价矩阵 {C.shape} 与供给 {len(s_val)} / 需求 {len(d_val)} 不匹配")

    diag = []
    C = C.copy()
    if s_val.sum() > d_val.sum() + 1e-9:            # 供 > 需：补虚拟销地
        gap = s_val.sum() - d_val.sum()
        C = np.hstack([C, np.zeros((C.shape[0], 1))])
        d_val = np.append(d_val, gap)
        d_names = d_names + ["虚拟销地"]
        diag.append(f"供大于需 {gap:.2f}，补虚拟销地（运价 0）")
    elif d_val.sum() > s_val.sum() + 1e-9:          # 需 > 供：补虚拟产地
        gap = d_val.sum() - s_val.sum()
        C = np.vstack([C, np.zeros((1, C.shape[1]))])
        s_val = np.append(s_val, gap)
        s_names = s_names + ["虚拟产地"]
        diag.append(f"需大于供 {gap:.2f}，补虚拟产地（运价 0）")
    else:
        diag.append("供需平衡，无需补虚拟节点")

    m, n = C.shape
    # 每个产地一行、每个销地一行：共 m+n 条等式约束，变量按行展平
    A_eq = np.zeros((m + n, m * n))
    for i in range(m):
        A_eq[i, i * n:(i + 1) * n] = 1
    for j in range(n):
        A_eq[m + j, j::n] = 1
    b_eq = np.concatenate([s_val, d_val])

    diag.append(f"决策变量 {m} × {n} = {m*n} 个，等式约束 {m+n} 条")
    return dict(c=C.ravel(), A_eq=A_eq, b_eq=b_eq,
                bounds=[(0, None)] * (m * n),
                s_names=s_names, d_names=d_names, cost=C, diag=diag,
                var_names=[f"{s_names[i]}→{d_names[j]}" for i in range(m) for j in range(n)])


def solve_transport(c, A_eq, b_eq, bounds, var_names=None):
    """解运输问题（本质是线性规划）。"""
    t0 = time.time()
    r = sp_opt.linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    x = r.x if r.x is not None else np.full(len(c), np.nan)
    return dict(x=x, objective=float(r.fun) if r.fun is not None else np.nan,
                ok=bool(r.success), message=str(r.message),
                var_names=var_names or [f"x{i}" for i in range(len(c))],
                elapsed=time.time() - t0)


def transport_tables(res, model):
    """把解整理成调运矩阵 + 明细表 + 摘要。"""
    m, n = model["cost"].shape
    X = res["x"].reshape(m, n)
    t_plan = pd.DataFrame(X, index=model["s_names"], columns=model["d_names"])
    t_plan.index.name = "产地\\销地"
    # 只保留实际运量 > 0 的线路，论文里更好读
    rows = []
    for i in range(m):
        for j in range(n):
            if X[i, j] > 1e-6:
                rows.append({"产地": model["s_names"][i], "销地": model["d_names"][j],
                             "运量": round(X[i, j], 3),
                             "运价": round(model["cost"][i, j], 3),
                             "运费": round(X[i, j] * model["cost"][i, j], 3)})
    t_detail = pd.DataFrame(rows)
    t_sum = pd.DataFrame([{
        "总运费": round(res["objective"], 3),
        "产地数": m, "销地数": n,
        "运输线路数": len(t_detail),
        "状态": "最优" if res["ok"] else "未求得最优",
    }])
    return t_plan, t_detail, t_sum


## 4. 示例 A：已处理矩阵


In [5]:
# =====================================================================
# 4. 示例 A：已处理矩阵（供、需、运价都齐了）
# =====================================================================
supply = {"工厂A": 100, "工厂B": 80}                 # 产量
demand = {"门店1": 60, "门店2": 70, "门店3": 50}      # 需求量
cost = pd.DataFrame([[4, 6, 8],
                     [5, 4, 3]],
                    index=list(supply), columns=list(demand))
print("运价矩阵（元/吨）：")
print(cost.to_string())

model_a = build_transport(supply, demand, cost)
print("\n--- 建模信息 ---")
for line in model_a["diag"]:
    print(" ·", line)

res_a = solve_transport(model_a["c"], model_a["A_eq"], model_a["b_eq"],
                        model_a["bounds"], model_a["var_names"])
t_plan_a, t_detail_a, t_sum_a = transport_tables(res_a, model_a)

print("\n=== 调运方案（行=产地，列=销地）===")
print(t_plan_a.round(2).to_string())
print("\n=== 非零运输线路 ===")
print(t_detail_a.to_string(index=False))
print("\n", t_sum_a.to_string(index=False))

# 核对：每个产地运出量 = 产量，每个销地收到量 = 需求量
print("\n产地核对：", dict(zip(model_a["s_names"], np.round(t_plan_a.sum(axis=1), 3))))
print("销地核对：", dict(zip(model_a["d_names"], np.round(t_plan_a.sum(axis=0), 3))))


运价矩阵（元/吨）：
     门店1  门店2  门店3
工厂A    4    6    8
工厂B    5    4    3

--- 建模信息 ---
 · 供需平衡，无需补虚拟节点
 · 决策变量 2 × 3 = 6 个，等式约束 5 条

=== 调运方案（行=产地，列=销地）===
        门店1   门店2   门店3
产地\销地                  
工厂A    60.0  40.0   0.0
工厂B     0.0  30.0  50.0

=== 非零运输线路 ===
 产地  销地   运量  运价    运费
工厂A 门店1 60.0 4.0 240.0
工厂A 门店2 40.0 6.0 240.0
工厂B 门店2 30.0 4.0 120.0
工厂B 门店3 50.0 3.0 150.0

   总运费  产地数  销地数  运输线路数 状态
750.0    2    3      4 最优

产地核对： {'工厂A': 100.0, '工厂B': 80.0}
销地核对： {'门店1': 60.0, '门店2': 70.0, '门店3': 50.0}


## 5. 示例 B：长表 → 矩阵（含不平衡）


In [6]:
# =====================================================================
# 5. 示例 B：长表 → 运价矩阵（原始数据模式）
# =====================================================================
# 附件常常是"产地-销地-运价"三列，另附两张供给/需求表。
long_table = pd.DataFrame({
    "产地": ["工厂A"] * 3 + ["工厂B"] * 3,
    "销地": ["门店1", "门店2", "门店3"] * 2,
    "运价": [4, 6, 8, 5, 4, 3],
})
print("原始长表：", long_table.shape)
print(long_table.to_string(index=False))

# 长表 → 矩阵（一行产地、一列销地）
cost_b = pivot_long(long_table, "产地", "销地", "运价", agg="first").set_index("产地")
cost_b = cost_b.astype(float)
print("\n转成运价矩阵：")
print(cost_b.to_string())

# 供需也可以从表里读；这里演示"需 > 供"的不平衡情形，看它怎么补虚拟产地
supply_b = pd.Series({"工厂A": 100, "工厂B": 60})
demand_b = pd.Series({"门店1": 60, "门店2": 70, "门店3": 50})
model_b = build_transport(supply_b, demand_b, cost_b)
print("\n--- 建模信息 ---")
for line in model_b["diag"]:
    print(" ·", line)

res_b = solve_transport(model_b["c"], model_b["A_eq"], model_b["b_eq"],
                        model_b["bounds"], model_b["var_names"])
t_plan_b, t_detail_b, t_sum_b = transport_tables(res_b, model_b)
print("\n=== 调运方案（含虚拟产地）===")
print(t_plan_b.round(2).to_string())
print("\n", t_sum_b.to_string(index=False))


原始长表： (6, 3)
 产地  销地  运价
工厂A 门店1   4
工厂A 门店2   6
工厂A 门店3   8
工厂B 门店1   5
工厂B 门店2   4
工厂B 门店3   3

转成运价矩阵：
     门店1  门店2  门店3
产地                
工厂A  4.0  6.0  8.0
工厂B  5.0  4.0  3.0

--- 建模信息 ---
 · 需大于供 20.00，补虚拟产地（运价 0）
 · 决策变量 3 × 3 = 9 个，等式约束 6 条

=== 调运方案（含虚拟产地）===
        门店1   门店2   门店3
产地\销地                  
工厂A    60.0  40.0   0.0
工厂B     0.0  10.0  50.0
虚拟产地    0.0  20.0   0.0

   总运费  产地数  销地数  运输线路数 状态
670.0    3    3      5 最优


## 6. 结果导出


In [7]:
# =====================================================================
# 结果导出：CSV + 多 Sheet Excel + 图
# =====================================================================
def export_result(tables, cfg, fig=None, fig_name="结果图"):
    """把结果写成文件，返回写出的路径。

    tables : {Sheet 名: DataFrame}，第一张建议放"摘要"
    cfg    : 配置字典，用 tag / make_plots / save_excel 三个键
    fig    : 传了 matplotlib 图对象就保存；cfg["make_plots"]=False 时连图都不生成
    """
    d = OUT_DIR / f"优化_{cfg['tag']}"
    d.mkdir(parents=True, exist_ok=True)
    written = {}

    first = next(iter(tables.values()))
    p_csv = d / f"{cfg['tag']}_主表.csv"
    first.to_csv(p_csv, index=False, encoding="utf-8-sig")
    written["主表"] = p_csv

    if cfg.get("save_excel", True):
        p_xlsx = d / f"{cfg['tag']}_结果汇总.xlsx"
        with pd.ExcelWriter(p_xlsx, engine="openpyxl") as w:
            for name, frame in tables.items():
                frame.to_excel(w, sheet_name=str(name)[:28], index=False)
        written["汇总"] = p_xlsx
        print(f"[表格] {p_xlsx.name}（{len(tables)} 个 Sheet）")

    if fig is not None and cfg.get("make_plots", True) and HAS_MPL:
        p_png = d / f"{cfg['tag']}_{fig_name}.png"
        fig.savefig(p_png, dpi=150, bbox_inches="tight")
        if "agg" not in matplotlib.get_backend().lower():   # 无界面后端不弹窗，也不报警告
            plt.show()
        plt.close(fig)          # 关掉画布：反复运行不会在内存里堆图
        written["图片"] = p_png
        print(f"[图片] {p_png.name}")

    print("[输出] ", d)
    return written


In [8]:
# =====================================================================
# 6. 结果导出与调运热力图
# =====================================================================
fig = None
if CFG["make_plots"] and HAS_MPL:
    M = t_plan_a.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(6, 3.8))
    im = ax.imshow(M, cmap="Blues", aspect="auto")
    ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(t_plan_a.columns)
    ax.set_yticks(range(M.shape[0])); ax.set_yticklabels(t_plan_a.index)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center",
                    color="#1f3d66", fontsize=11)
    ax.set_title(f"最优调运方案（总运费 {res_a['objective']:.0f}）")
    fig.colorbar(im, ax=ax, shrink=.85)
    fig.tight_layout()

written = export_result(
    {"结果摘要": t_sum_a, "调运方案": t_plan_a.reset_index(),
     "运输明细": t_detail_a, "运价矩阵": cost.reset_index()},
    CFG, fig=fig, fig_name="调运方案")
print(written)


[表格] 运输方案_结果汇总.xlsx（4 个 Sheet）
[图片] 运输方案_调运方案.png
[输出]  D:\git项目\输出数据\优化_运输方案
{'主表': WindowsPath('D:/git项目/输出数据/优化_运输方案/运输方案_主表.csv'), '汇总': WindowsPath('D:/git项目/输出数据/优化_运输方案/运输方案_结果汇总.xlsx'), '图片': WindowsPath('D:/git项目/输出数据/优化_运输方案/运输方案_调运方案.png')}


换题目时按三步走：

1. 数据放进 `导入数据/`（csv / xlsx 都行）；
2. 改本 Notebook 第 1 个代码单元里的 `CFG`：文件名、列名映射、方向、输出名；
3. `Run All`，结果写到 `输出数据/优化_{tag}/`，同名覆盖，不会越跑越多。

**两种输入方式怎么选**

- 数据还脏（有空行、重复、999、文本数字、列名带空格）→ 走「原始数据」示例，让 `clean_frame` 处理；
- 数据已经清洗/标准化完（比如上一节 `数据学习.ipynb` 输出的 `final_cleaned.csv`）→ 走「已处理数据」示例，直接给矩阵或宽表。

不要混：**同一次建模只用一种入口**，否则等于把同一份数据预处理了两遍。
